In [16]:
import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import datetime
import os
import sys
from scipy.stats import spearmanr

In [ ]:
# [자동 경로 설정] 프로젝트 루트를 찾아 데이터를 연결합니다.
def get_root():
    path = os.getcwd()
    while not os.path.exists(os.path.join(path, 'data')) and path != os.path.dirname(path):
        path = os.path.dirname(path)
    return path

ROOT = get_root()
B2_PATH = os.path.join(ROOT, 'data/processed/B2_FOOD_POS_SALE.parquet')
B4_PATH = os.path.join(ROOT, 'data/processed/B4_CLEAN_FOOD_ITEM.parquet')

print(f"✅ 프로젝트 루트 감지: {ROOT}")

# 1. 데이터 로드
def load_data(path):
    if os.path.exists(path):
        return pl.scan_parquet(path)
    else:
        print(f"❌ 파일을 찾을 수 없습니다: {path}")
        return None

b2_lazy = load_data(B2_PATH)
b4_lazy = load_data(B4_PATH)

✅ 프로젝트 루트 감지: /Users/yumi/Seven-eleven


In [18]:
# 🚀 [데이터 결합] 판매 데이터(B2)와 상품 마스터(B4)를 하나로 묶습니다.
if 'b2_lazy' in globals() and 'b4_lazy' in globals():
    print("📊 상품별 매출 집계 및 마스터 데이터 결합 중... ")
    
    # 1. 판매 데이터를 상품별로 요약
    df_item_sales = (
        b2_lazy.group_by("상품코드")
        .agg([
            pl.col("판매금액").sum().alias("총매출액"),
            pl.col("판매수량").sum().alias("총판매량")
        ])
    )

    # 2. 요약된 판매 데이터와 상품 마스터 결합
    df_merged = (
        df_item_sales.join(
            b4_lazy, 
            left_on="상품코드", 
            right_on="ITEM_CD", 
            how="inner"
        )
    ).collect() 

    print(f"✅ 결합 완료! 분석 대상 식품 상품 수: {len(df_merged):,}개")
else:
    print("❌ 데이터 로드 셀을 먼저 실행해 주세요.")

📊 상품별 매출 집계 및 마스터 데이터 결합 중... 
✅ 결합 완료! 분석 대상 식품 상품 수: 10,604개


In [21]:
if 'df_merged' in globals():
    total_sales_amt = df_merged["총매출액"].sum()
    threshold_q95 = df_merged["총매출액"].quantile(0.95)
    df_top5 = df_merged.filter(pl.col("총매출액") >= threshold_q95).sort("총매출액", descending=True)
    concentration_ratio = (df_top5["총매출액"].sum() / total_sales_amt) * 100
    
    print(f"📊 [전체 분석 결과]")
    print(f"- 상위 5% 매출 비중: {concentration_ratio:.2f}%")
    
    output_excel_path = os.path.join(ROOT, 'eda/ipynb/yumi/top_5_percent_products.xlsx')
    df_top5.to_pandas().to_excel(output_excel_path, index=False)
    print(f"💾 저장 경로: {output_excel_path}")
    display(df_top5.select(["ITEM_NM", "ITEM_MDDV_NM", "총매출액"]).head(10))

📊 [전체 분석 결과]
- 상위 5% 매출 비중: 53.45%
💾 저장 경로: /Users/yumi/Seven-eleven/eda/ipynb/yumi/top_5_percent_products.xlsx


ITEM_NM,ITEM_MDDV_NM,총매출액
str,str,i64
"""빙그레)바나나우유240ml""","""가공우유""",1636959147
"""오비)카스500ml캔""","""국산맥주""",1516584480
"""마루카네)감동란2입""","""냉장간식""",1443745205
"""칠성)아이시스500ml(8.0)""","""생수""",1273662818
"""오츠카)포카리스웨트620ml""","""스포츠음료""",1138293946
"""광동)제주삼다수500ml""","""생수""",1122562650
"""진로)참이슬후레쉬640ml페트""","""소주""",879030700
"""서울탁)장수생막걸리750ml""","""막걸리""",770282280
"""HK)헛개컨디션100ml""","""기능성드링크""",701281899


In [ ]:
if 'df_merged' in globals():
    print("📊 중분류별 상위 5% 매출 비중 분석 중...")
    results = []
    top_5_items_all = []
    small_categories = df_merged["ITEM_SMDV_NM"].unique().to_list()
    
    for cat in small_categories:
        df_cat = df_merged.filter(pl.col("ITEM_SMDV_NM") == cat)
        if len(df_cat) < 5: continue # 상품 수가 너무 적은 카테고리는 제외
        
        total_cat_sales = df_cat["총매출액"].sum()
        threshold = df_cat["총매출액"].quantile(0.95)
        top_5_in_cat = df_cat.filter(pl.col("총매출액") >= threshold).sort("총매출액", descending=True)
        
        concentration_share = (top_5_in_cat["총매출액"].sum() / total_cat_sales * 100) if total_cat_sales > 0 else 0
        
        results.append({
            "소분류명": cat,
            "전체상품수": len(df_cat),
            "상위5%상품수": len(top_5_in_cat),
            "상위5%매출비중(%)": concentration_share
        })
        top_5_items_all.append(top_5_in_cat.with_columns([pl.lit(concentration_share).alias("카테고리내_비중")]))

    df_summary = pl.DataFrame(results).sort("상위5%매출비중(%)", descending=True)
    df_final_list = pl.concat(top_5_items_all)
    
    output_path = os.path.join(ROOT, 'eda/ipynb/yumi/top_5_by_small_category.xlsx')
    with pd.ExcelWriter(output_path) as writer:
        df_summary.to_pandas().to_excel(writer, sheet_name='요약', index=False)
        df_final_list.to_pandas().to_excel(writer, sheet_name='상세리스트', index=False)
    
    print(f"✅ 소분류별 분석 완료! 엑셀 저장됨: {output_path}")
    display(df_summary.head(10))
else:
    print("❌ 'df_merged' 변수를 찾을 수 없습니다.")

📊 소분류별 상위 5% 매출 비중 분석 중...
✅ 소분류별 분석 완료! 엑셀 저장됨: /Users/yumi/Seven-eleven/eda/ipynb/yumi/top_5_by_small_category.xlsx


소분류명,전체상품수,상위5%상품수,상위5%매출비중(%)
str,i64,i64,f64
"""조미세트""",15,2,104.294518
"""청주""",10,1,97.48889
"""푸딩""",13,2,95.641139
"""증류식소주""",35,3,91.988741
"""소스/양념류""",17,2,85.068808
"""생막걸리""",20,2,84.281219
"""바나나우유""",15,2,83.823659
"""카톤""",23,2,83.323706
"""기타류 """,16,2,83.262082


In [26]:
# 🚀 [소분류별 상위 5% 분석: 카테고리별 그룹화 + 상품명 리스트]
if 'df_merged' in globals():
    print("📊 소분류별 상위 5% 히트 상품 분석 중...")
    
    summary_results = []
    
    # 1. 소분류(ITEM_SMDV_NM)별로 유니크 리스트 추출
    small_categories = df_merged["ITEM_SMDV_NM"].unique().to_list()
    
    for cat in small_categories:
        # 해당 소분류 데이터만 필터링
        df_cat = df_merged.filter(pl.col("ITEM_SMDV_NM") == cat)
        
        if len(df_cat) == 0: continue
        
        # 2. 해당 카테고리 내 매출 상위 5% 임계값 계산 및 상품 추출
        threshold = df_cat["총매출액"].quantile(0.95)
        df_top5 = df_cat.filter(pl.col("총매출액") >= threshold).sort("총매출액", descending=True)
        
        # 3. 상위 5% 상품명 리스트 작성 (콤마로 연결)
        top5_names = df_top5["ITEM_NM"].to_list()
        # 가독성을 위해 리스트를 문자열로 변환
        names_str = ", ".join(top5_names)
        
        # 4. 요약 정보 수집
        total_sales = df_cat["총매출액"].sum()
        top5_sales = df_top5["총매출액"].sum()
        
        summary_results.append({
            "소분류명": cat,
            "전체상품수": len(df_cat),
            "상위5%상품수": len(df_top5),
            "카테고리총매출": total_sales,
            "상위5%매출비중(%)": (top5_sales / total_sales * 100) if total_sales > 0 else 0,
            "상위5%상품명리스트": names_str
        })
    
    # 5. 데이터프레임 생성 및 정렬 (비중이 높은 순)
    df_final = pl.DataFrame(summary_results).sort("상위5%매출비중(%)", descending=True)
    
    # 6. 엑셀 파일 저장 (yumi 폴더 내)
    # ROOT 변수가 정의되어 있어야 합니다.
    output_path = os.path.join(ROOT, 'eda/ipynb/yumi/category_top5_with_names.xlsx')
    
    # 엑셀 저장을 위해 pandas로 변환 후 저장 (openpyxl 또는 xlsxwriter 필요)
    df_final.to_pandas().to_excel(output_path, index=False)
    
    print(f"\n✅ 분석 완료! 결과가 엑셀로 저장되었습니다.")
    print(f"💾 저장 경로: {output_path}")
    
    # 상위 20개 카테고리 결과 출력
    display(df_final.head(10))
else:
    print("❌ 'df_merged' 변수가 없습니다. 데이터 결합 셀을 먼저 실행해 주세요.")

📊 소분류별 상위 5% 히트 상품 분석 중...

✅ 분석 완료! 결과가 엑셀로 저장되었습니다.
💾 저장 경로: /Users/yumi/Seven-eleven/eda/ipynb/yumi/category_top5_with_names.xlsx


소분류명,전체상품수,상위5%상품수,카테고리총매출,상위5%매출비중(%),상위5%상품명리스트
str,i64,i64,i64,f64,str
"""조미세트""",15,2,6041423,104.294518,"""26설)대상_고급유6호, 25추_LG)프리미엄햄복합1호"""
"""브랜디""",1,1,9019900,100.0,"""MH)헤네시VSOP500ml"""
"""빅바이트""",3,1,2200,100.0,"""존슨빌)폴리쉬소시지108g"""
"""택배선물세트""",2,1,152000,100.0,"""죽력고32도700ml"""
"""버터""",1,1,910020,100.0,"""동원)모닝버터해바리기유200g"""
"""냉장주스(대)""",1,1,51847177,100.0,"""칠성)콜드오렌지900ml"""
"""꿀""",1,1,1371975,100.0,"""꽃샘)야생화꿀300g"""
"""모듬안주""",1,1,3109532,100.0,"""정화)사각모듬안주109g"""
"""모닝빵샌드""",1,1,2900,100.0,"""롯데)소프트에그롤"""
